<a href="https://colab.research.google.com/github/Maryam-Taherzadeh/Computational-Drug-Discovery/blob/main/project-01-predict-ic50-mdm2-p53/notebooks/mdm2_part3_padel_descriptor_dataset_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🧪 Part 3: Descriptor Calculation and Dataset Preparation

**Author: Maryam Taherzadeh**

In this section, we continue the computational drug discovery pipeline by generating molecular descriptors and preparing the dataset for machine learning.

This step is inspired by the work of Chanin Nantasenamat ("Data Professor") and focuses on transforming chemical structures into numerical representations suitable for modeling.

---

### Objective

- Calculate molecular descriptors for each compound  
- Convert chemical structures into numerical features  
- Prepare a clean dataset for machine learning models  

---

###  Why this is important

Machine learning models cannot understand SMILES strings directly.

 We need to convert molecules into **numerical features (descriptors)** such as:
- Molecular Weight (MW)  
- LogP (lipophilicity)  
- Number of hydrogen bond donors  
- Number of hydrogen bond acceptors  
- Molecular fingerprints (e.g., ECFP)  

These descriptors allow the model to learn relationships between molecular structure and biological activity (pIC50).

---

###  What we will do

1. Calculate descriptors using:
   - RDKit (basic descriptors)  
   - PaDEL (advanced descriptors & fingerprints)  

2. Combine descriptors with bioactivity data (pIC50)  

3. Clean and preprocess the dataset:
   - Remove missing values  
   - Normalize features (if needed)  

4. Save the final dataset for modeling (Part 4)  

---

###  Output

- Feature matrix (X)  
- Target variable (y = pIC50)  
- Machine learning-ready dataset  

---

###  Next Step

In Part 4, we will:
- Train machine learning models (Random Forest, XGBoost)  
- Evaluate performance  
- Predict bioactivity of new compounds  

## **Download PaDEL-Descriptor**

In [19]:
# Install conda in Colab
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [25]:
!conda create -y -n padel_env -c conda-forge python=3.10 padel

Channels:
 - conda-forge
Platform: linux-64
Solving environment: / - done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/padel_env

  added / updated specs:
    - padel
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    alsa-lib-1.2.15.3          |       hb03c661_0         571 KB  conda-forge
    bzip2-1.0.8                |       hda65f42_9         254 KB  conda-forge
    ca-certificates-2026.4.22  |       hbd8a1cb_0         128 KB  conda-forge
    cairo-1.18.4               |       he90730b_1         966 KB  conda-forge
    font-ttf-dejavu-sans-mono-2.37|       hab24e00_0         388 KB 

In [26]:
!conda run -n padel_env bash -lc 'find $CONDA_PREFIX -name "*PaDEL*.jar"'

/usr/local/envs/padel_env/PaDEL-Descriptor.jar
/usr/local/envs/padel_env/lib/libPaDEL.jar
/usr/local/envs/padel_env/lib/libPaDEL-Jobs.jar
/usr/local/envs/padel_env/lib/libPaDEL-Descriptor.jar



In [27]:
!which padel
!which padeldescriptor
!conda list | grep padel
# !padel -h
!find /usr/local -name "*PaDEL*.jar"

/usr/local/pkgs/padel-2.21-py310hff52083_5/PaDEL-Descriptor.jar
/usr/local/pkgs/padel-2.21-py310hff52083_5/lib/libPaDEL.jar
/usr/local/pkgs/padel-2.21-py310hff52083_5/lib/libPaDEL-Jobs.jar
/usr/local/pkgs/padel-2.21-py310hff52083_5/lib/libPaDEL-Descriptor.jar
/usr/local/envs/padel_env/PaDEL-Descriptor.jar
/usr/local/envs/padel_env/lib/libPaDEL.jar
/usr/local/envs/padel_env/lib/libPaDEL-Jobs.jar
/usr/local/envs/padel_env/lib/libPaDEL-Descriptor.jar


## **Load bioactivity data**

Download the curated ChEMBL bioactivity data that has been pre-processed from Parts 1 and 2 of this Bioinformatics Project series. Here we will be using the **bioactivity_data_3class_pIC50.csv** file that essentially contain the pIC50 values that we will be using for building a regression model.

In [28]:

from google.colab import files
uploaded = files.upload()


Saving my_bioactivity_data.csv to my_bioactivity_data (1).csv


In [29]:
import pandas as pd

In [30]:
df3 = pd.read_csv('my_bioactivity_data.csv')

In [31]:
df3

,molecule_chembl_id,canonical_smiles,bioactivity_class,mol,MW,LogP,NumHDonors,NumHAcceptors,pIC50
0,CHEMBL178578,O=C(O)[C@H](c1ccccc1)N1C(=O)c2cc(I)ccc2NC(=O)[...,intermediate,<rdkit.Chem.rdchem.Mol object at 0x7e8f8cc297e0>,580.300,5.27150,2,3,5.769551
1,CHEMBL361103,O=C(O)[C@H](c1ccc(Cl)cc1)N1C(=O)c2cc(I)ccc2NC(...,active,<rdkit.Chem.rdchem.Mol object at 0x7e8f8cc299a0>,581.193,5.55950,2,3,6.698970
2,CHEMBL182051,C[C@H](c1ccc(Cl)cc1)N(C(=O)/C=C/c1ccccc1)[C@H]...,inactive,<rdkit.Chem.rdchem.Mol object at 0x7e8f8cc29a10>,453.369,5.82300,1,2,4.522879
3,CHEMBL360540,C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2)N(CCCC...,inactive,<rdkit.Chem.rdchem.Mol object at 0x7e8f8cc29a80>,551.470,6.76250,1,3,4.886057
4,CHEMBL427316,C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2Br)N(CC...,intermediate,<rdkit.Chem.rdchem.Mol object at 0x7e8f8cc29af0>,630.366,7.52500,1,3,5.443697
...,...,...,...,...,...,...,...,...,...
5787,CHEMBL6014971,COc1ncc(-c2nc3c(n2C(C)C)[C@H](c2ccc(Cl)cc2)N(c...,active,<rdkit.Chem.rdchem.Mol object at 0x7e8fb4151380>,568.417,5.72350,0,8,9.619789
5788,CHEMBL5805098,COc1ncc(-c2nc3c(n2C(C)C)[C@H](c2ccc(Cl)cc2)N(c...,active,<rdkit.Chem.rdchem.Mol object at 0x7e8fb41513f0>,547.999,5.37852,0,8,9.161151
5789,CHEMBL6045300,COc1ncc(-c2nc3c(n2C(C)C)[C@@H](c2ccc(Cl)cc2)N(...,active,<rdkit.Chem.rdchem.Mol object at 0x7e8fb4151460>,565.421,5.03750,0,8,8.543634
5790,CHEMBL6055628,COc1ncc(-c2nc3c(n2C(C)C)[C@@H](c2ccc(Cl)cc2)N(...,active,<rdkit.Chem.rdchem.Mol object at 0x7e8fb41514d0>,564.433,5.64250,0,7,9.602060


In [32]:
selection = ['canonical_smiles','molecule_chembl_id']
df3_selection = df3[selection]
df3_selection.to_csv('molecule.smi', sep='\t', index=False, header=False)

In [33]:
! cat molecule.smi | head -5

O=C(O)[C@H](c1ccccc1)N1C(=O)c2cc(I)ccc2NC(=O)[C@@H]1c1ccc(C(F)(F)F)cc1	CHEMBL178578
O=C(O)[C@H](c1ccc(Cl)cc1)N1C(=O)c2cc(I)ccc2NC(=O)[C@@H]1c1ccc(Cl)cc1	CHEMBL361103
C[C@H](c1ccc(Cl)cc1)N(C(=O)/C=C/c1ccccc1)[C@H](C(N)=O)c1ccc(Cl)cc1	CHEMBL182051
C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2)N(CCCCC(=O)O)C(=O)[C@@H]1c1ccc(Cl)cc1	CHEMBL360540
C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2Br)N(CCCCC(=O)O)C(=O)[C@@H]1c1ccc(Cl)cc1	CHEMBL427316


In [34]:
! cat molecule.smi | wc -l

5792


## **Calculate fingerprint descriptors**


### **Calculate PaDEL descriptors**

In [38]:
!java -jar /usr/local/envs/padel_env/PaDEL-Descriptor.jar \
 -removesalt \
 -standardizenitro \
 -fingerprints \
 -dir ./ \
 -file descriptors_output.csv

Streaming output truncated to the last 5000 lines.
Processing CHEMBL3318780 in molecule.smi (794/5792). Average speed: 0.16 s/mol.
Processing CHEMBL3318781 in molecule.smi (795/5792). Average speed: 0.16 s/mol.
Processing CHEMBL3318782 in molecule.smi (796/5792). Average speed: 0.16 s/mol.
Processing CHEMBL3318783 in molecule.smi (797/5792). Average speed: 0.16 s/mol.
Processing CHEMBL3318784 in molecule.smi (798/5792). Average speed: 0.16 s/mol.
Processing CHEMBL3318785 in molecule.smi (799/5792). Average speed: 0.16 s/mol.
Processing CHEMBL3318786 in molecule.smi (800/5792). Average speed: 0.16 s/mol.
Processing CHEMBL3318787 in molecule.smi (801/5792). Average speed: 0.16 s/mol.
Processing CHEMBL2398246 in molecule.smi (802/5792). Average speed: 0.16 s/mol.
Processing CHEMBL3402515 in molecule.smi (803/5792). Average speed: 0.16 s/mol.
Processing CHEMBL3402516 in molecule.smi (804/5792). Average speed: 0.16 s/mol.
Processing CHEMBL3402517 in molecule.smi (805/5792). Average speed: 0

In [42]:
import pandas as pd

df_padel = pd.read_csv('descriptors_output.csv')
df_padel.head()


,Name,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,CHEMBL178578,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CHEMBL361103,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CHEMBL182051,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CHEMBL360540,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CHEMBL427316,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [43]:
df_padel.shape

(5792, 882)

In [44]:
! cat molecule.smi | head -5

O=C(O)[C@H](c1ccccc1)N1C(=O)c2cc(I)ccc2NC(=O)[C@@H]1c1ccc(C(F)(F)F)cc1	CHEMBL178578
O=C(O)[C@H](c1ccc(Cl)cc1)N1C(=O)c2cc(I)ccc2NC(=O)[C@@H]1c1ccc(Cl)cc1	CHEMBL361103
C[C@H](c1ccc(Cl)cc1)N(C(=O)/C=C/c1ccccc1)[C@H](C(N)=O)c1ccc(Cl)cc1	CHEMBL182051
C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2)N(CCCCC(=O)O)C(=O)[C@@H]1c1ccc(Cl)cc1	CHEMBL360540
C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2Br)N(CCCCC(=O)O)C(=O)[C@@H]1c1ccc(Cl)cc1	CHEMBL427316


In [45]:
! cat molecule.smi | wc -l

5792


## **Preparing the X and Y Data Matrices**

### **X data matrix**

In [46]:
df3_X = pd.read_csv('descriptors_output.csv')

In [47]:
df3_X

,Name,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,CHEMBL178578,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CHEMBL361103,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CHEMBL182051,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CHEMBL360540,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CHEMBL427316,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5787,CHEMBL5805098,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5788,CHEMBL6014971,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5789,CHEMBL6045300,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5790,CHEMBL6055628,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [48]:
df3_X = df3_X.drop(columns=['Name'])
df3_X

,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,PubchemFP9,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,1,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5787,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
5788,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
5789,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
5790,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


## **Y variable**

### **Convert IC50 to pIC50**

In [49]:
df3_Y = df3['pIC50']
df3_Y

,pIC50
0,5.769551
1,6.698970
2,4.522879
3,4.886057
4,5.443697
...,...
5787,9.619789
5788,9.161151
5789,8.543634
5790,9.602060


## **Combining X and Y variable**

In [50]:
dataset3 = pd.concat([df3_X,df3_Y], axis=1)
dataset3

,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,PubchemFP9,...,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880,pIC50
0,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,5.769551
1,1,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,6.698970
2,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,4.522879
3,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,4.886057
4,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,5.443697
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5787,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,9.619789
5788,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,9.161151
5789,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,8.543634
5790,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,9.602060


In [52]:
dataset3.to_csv('mdm2_07_padel_descriptors_pIC50.csv', index=False)

# **Let's download the CSV file to your local computer for the Part 3B (Model Building).**